In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 3.4 Evaluating Tree-Based Models
- Precision-Recall curves
- Confusion matrices

## Setup

In [ ]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix

RANDOM_STATE = 42
colors = ['#2ecc71', '#3498db', '#e74c3c']

train_df = pd.read_csv('../data/training.csv')
test_df = pd.read_csv('../data/testing.csv')
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

train_enc = pd.get_dummies(train_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

# Impute with TRAIN medians only, never test's own (avoids leakage)
train_medians = train_enc.median()
train_enc = train_enc.fillna(train_medians)
test_enc = test_enc.fillna(train_medians)

X_test, y_test = test_enc, test_df['DEPARTED']

# Load the tuned models saved by 3.3_lesson (not retrained here)
models = {
    'Decision Tree': joblib.load('../models/dt_tuned_f1.pkl'),
    'Random Forest': joblib.load('../models/rf_tuned_f1.pkl'),
    'XGBoost': joblib.load('../models/xgb_tuned_f1.pkl'),
}

predictions, probabilities = {}, {}
for name, model in models.items():
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)[:, 1]

print("Loaded tuned models from 3.3_lesson and generated predictions.")

## Precision-Recall Curves

In [ ]:
fig = go.Figure()
for i, (name, prob) in enumerate(probabilities.items()):
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines',
        name=f'{name} (AP={ap:.3f})', line=dict(color=colors[i], width=2)))

prevalence = y_test.mean()
fig.add_hline(y=prevalence, line_dash='dash', line_color='gray',
              annotation_text=f'Baseline ({prevalence:.1%})')

fig.update_layout(title='Precision-Recall Curves', height=500,
    xaxis_title='Recall', yaxis_title='Precision')
fig.show()

## Confusion Matrices

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(models.keys()))

for col, (name, pred) in enumerate(predictions.items(), 1):
    cm = confusion_matrix(y_test, pred)
    labels = [['TN', 'FP'], ['FN', 'TP']]
    text = [[f'{labels[i][j]}<br>{cm[i][j]}' for j in range(2)] for i in range(2)]

    fig.add_trace(go.Heatmap(z=cm, text=text, texttemplate='%{text}',
        colorscale='Blues', showscale=False,
        x=['Predicted Enrolled', 'Predicted Departed'],
        y=['Actual Enrolled', 'Actual Departed']), row=1, col=col)

fig.update_layout(title='Confusion Matrices', height=400)
fig.show()

## Summary - Performance Metrics in the Context of Imbalanced Data

When working with imbalanced datasets, standard metrics like accuracy can be misleading. For instance, a model predicting the majority class for all instances might achieve high accuracy but be useless for the minority class, which is often the class of interest.

### Precision-Recall (PR) Curves
PR curves offer a more informative view of model performance on imbalanced data, especially when the positive class is the minority. Unlike ROC curves, which can paint an overly optimistic picture for highly imbalanced datasets, PR curves focus directly on the trade-off between Precision and Recall.

### Confusion Matrices
Confusion matrices provide a detailed breakdown of correct and incorrect classifications. For imbalanced data, examining the raw counts of True Positives, False Positives, False Negatives, and True Negatives is invaluable for identifying where the model struggles, especially concerning the minority class.

**Next:** Module 4 — Model Comparison